# Chapter 13 - Introducing Automatic Optimization: Let's Build a Deep Learning Framework

## Summary
Writing forward and backward code by hand for every network is slow and error-prone. Frameworks (PyTorch, TensorFlow) solve this with **automatic differentiation**. This chapter builds a tiny one:
1. **Tensor** - a NumPy array that remembers *how it was created* (its `creators` and the operation).
2. **Autograd** - calling `.backward()` walks the computation graph and sends gradients to every creator using each operation's local derivative.
3. **More operations** - add, sub, mul, neg, sum, expand, transpose, matrix multiplication.
4. **Layers** (`Linear`, `Sequential`, activation layers) and **loss layers** (`MSELoss`, `CrossEntropyLoss`).
5. **Optimizer** (`SGD`) - updates the parameters using their gradients and then clears them.
6. **Embedding layer** and **index_select** so word indices can flow through the graph.
7. **Nonlinearity layers** (Tanh, Sigmoid, ReLU) and the **RNN cell** built from these building blocks.

(The book tracks gradient arrival with child counters. Here the graph is walked in **topological order**, which gives the same gradients and is easier to verify.)

## Theory
- **Computation graph:** every operation produces a new tensor that points to its inputs. Forward pass builds the graph.
- **Reverse-mode autodiff:** start with `dL/dL = 1` and apply the **chain rule** backward, one node at a time; each node sends `grad * local_derivative` to its inputs.
- **Gradient accumulation:** if a tensor is used in several places (e.g. shared weights in an RNN), the gradients from all uses are **summed**.
- **Local derivatives used here:**

| Op | Gradient to inputs |
|---|---|
| `a + b` | `g`, `g` |
| `a - b` | `g`, `-g` |
| `a * b` | `g*b`, `g*a` |
| `a.mm(b)` | `g b^T`, `a^T g` |
| `sum(dim)` | `g` broadcast back |
| `expand(dim)` | `g.sum(dim)` |
| `sigmoid` | `g * s(1-s)` |
| `tanh` | `g * (1 - t^2)` |
| `relu` | `g * (x > 0)` |
| `index_select` | scatter-add of `g` into the selected rows |
| softmax cross-entropy | `(softmax - onehot)/N` |

- **Why it matters:** with autograd you only write the forward pass; the backward pass is free.

In [1]:
# If packages are missing in this kernel, uncomment:
# %pip install numpy scikit-learn matplotlib
import numpy as np

In [2]:
# ---------- Mini deep-learning framework (autograd Tensor + layers + optimizer) ----------
import numpy as np

def _acc(t, g):
    # accumulate gradient g into tensor t (only if it tracks gradients)
    if t.autograd:
        t.grad = g if t.grad is None else t.grad + g

class Tensor:
    def __init__(self, data, autograd=False, creators=None, op=None):
        self.data = np.asarray(data, dtype=float)
        self.autograd, self.creators, self.op = autograd, creators, op
        self.grad, self.extra = None, {}

    def _new(self, data, creators, op, **kw):
        ag = any(c.autograd for c in creators)
        t = Tensor(data, autograd=ag, creators=creators if ag else None, op=op)
        t.extra = kw
        return t

    # ----- operations (each records how to send gradients back) -----
    def __add__(self, o): return self._new(self.data + o.data, [self, o], "add")
    def __sub__(self, o): return self._new(self.data - o.data, [self, o], "sub")
    def __mul__(self, o): return self._new(self.data * o.data, [self, o], "mul")
    def __neg__(self):    return self._new(-self.data, [self], "neg")
    def sum(self, dim):   return self._new(self.data.sum(dim), [self], "sum", dim=dim)
    def expand(self, dim, copies):
        data = np.repeat(np.expand_dims(self.data, dim), copies, axis=dim)
        return self._new(data, [self], "expand", dim=dim)
    def transpose(self):  return self._new(self.data.T, [self], "transpose")
    def mm(self, o):      return self._new(self.data.dot(o.data), [self, o], "mm")
    def sigmoid(self):    return self._new(1 / (1 + np.exp(-self.data)), [self], "sigmoid")
    def tanh(self):       return self._new(np.tanh(self.data), [self], "tanh")
    def relu(self):       return self._new(np.maximum(self.data, 0), [self], "relu")
    def index_select(self, idx):
        idx = np.asarray(idx.data if isinstance(idx, Tensor) else idx, dtype=int)
        return self._new(self.data[idx], [self], "index_select", idx=idx)
    def cross_entropy(self, target):
        t = np.asarray(target.data if isinstance(target, Tensor) else target, dtype=int)
        e = np.exp(self.data - self.data.max(1, keepdims=True))
        soft = e / e.sum(1, keepdims=True)
        loss = -np.log(soft[np.arange(len(t)), t] + 1e-12).mean()
        return self._new(loss, [self], "cross_entropy", soft=soft, t=t)

    # ----- backpropagation: topological order, then local gradient rules -----
    def backward(self, grad=None):
        order, seen = [], set()
        def visit(t):
            if id(t) in seen or not t.autograd: return
            seen.add(id(t))
            for c in (t.creators or []): visit(c)
            order.append(t)
        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else grad
        for t in reversed(order):
            if t.creators is not None: t._propagate()

    def _propagate(self):
        g, c, op = self.grad, self.creators, self.op
        if op == "add":   _acc(c[0], g); _acc(c[1], g)
        elif op == "sub": _acc(c[0], g); _acc(c[1], -g)
        elif op == "mul": _acc(c[0], g * c[1].data); _acc(c[1], g * c[0].data)
        elif op == "neg": _acc(c[0], -g)
        elif op == "sum": _acc(c[0], np.broadcast_to(np.expand_dims(g, self.extra["dim"]), c[0].data.shape).copy())
        elif op == "expand": _acc(c[0], g.sum(self.extra["dim"]))
        elif op == "transpose": _acc(c[0], g.T)
        elif op == "mm":  _acc(c[0], g.dot(c[1].data.T)); _acc(c[1], c[0].data.T.dot(g))
        elif op == "sigmoid": _acc(c[0], g * self.data * (1 - self.data))
        elif op == "tanh":    _acc(c[0], g * (1 - self.data ** 2))
        elif op == "relu":    _acc(c[0], g * (c[0].data > 0))
        elif op == "index_select":
            gr = np.zeros_like(c[0].data); np.add.at(gr, self.extra["idx"], g); _acc(c[0], gr)
        elif op == "cross_entropy":
            d = self.extra["soft"].copy(); d[np.arange(len(d)), self.extra["t"]] -= 1
            _acc(c[0], d / len(d) * g)

    def __repr__(self): return f"Tensor({self.data})"

# ---------- Layers ----------
class Layer:
    def __init__(self): self.parameters = []
    def get_parameters(self): return self.parameters

class Linear(Layer):
    def __init__(self, n_in, n_out, bias=True):
        super().__init__()
        self.weight = Tensor(np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in), autograd=True)
        self.bias = Tensor(np.zeros(n_out), autograd=True) if bias else None
        self.parameters = [self.weight] + ([self.bias] if bias else [])
    def forward(self, x):
        out = x.mm(self.weight)
        return out + self.bias.expand(0, len(x.data)) if self.bias is not None else out

class Sequential(Layer):
    def __init__(self, layers): super().__init__(); self.layers = layers
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def get_parameters(self): return [p for l in self.layers for p in l.get_parameters()]

class Tanh(Layer):
    def forward(self, x): return x.tanh()
class Sigmoid(Layer):
    def forward(self, x): return x.sigmoid()
class ReLU(Layer):
    def forward(self, x): return x.relu()

class Embedding(Layer):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)
        self.parameters = [self.weight]
    def forward(self, idx): return self.weight.index_select(idx)

class MSELoss(Layer):
    def forward(self, pred, target): return ((pred - target) * (pred - target)).sum(0)
class CrossEntropyLoss(Layer):
    def forward(self, pred, target_idx): return pred.cross_entropy(target_idx)

class SGD:
    def __init__(self, parameters, alpha=0.1): self.parameters, self.alpha = parameters, alpha
    def step(self, zero=True):
        for p in self.parameters:
            if p.grad is not None:
                p.data -= p.grad * self.alpha
                if zero: p.grad = None

### 1) Autograd in action

In [3]:
a = Tensor([1, 2, 3, 4, 5], autograd=True)
b = Tensor([2, 2, 2, 2, 2], autograd=True)
c = Tensor([5, 4, 3, 2, 1], autograd=True)
d = a + b
e = b + c
f = d + e
f.backward(np.array([1, 1, 1, 1, 1.]))
print("a.grad:", a.grad)
print("b.grad:", b.grad, "(used twice -> gradient summed)")
print("c.grad:", c.grad)

a.grad: [1. 1. 1. 1. 1.]
b.grad: [2. 2. 2. 2. 2.] (used twice -> gradient summed)
c.grad: [1. 1. 1. 1. 1.]


### 2) Check autograd against numerical gradients (gradient checking)

In [4]:
np.random.seed(0)
x = Tensor(np.random.randn(4, 3))
W = Tensor(np.random.randn(3, 2), autograd=True)
tgt = Tensor(np.random.randn(4, 2))
def loss_of(Wdata):
    W2 = Tensor(Wdata, autograd=True)
    diff = x.mm(W2).tanh() - tgt
    return float((diff * diff).data.sum())
out = (x.mm(W).tanh() - tgt) * (x.mm(W).tanh() - tgt)
out.sum(0).sum(0).backward()
num = np.zeros_like(W.data); eps = 1e-5
for i in range(W.data.shape[0]):
    for j in range(W.data.shape[1]):
        Wp, Wm = W.data.copy(), W.data.copy(); Wp[i, j] += eps; Wm[i, j] -= eps
        num[i, j] = (loss_of(Wp) - loss_of(Wm)) / (2 * eps)
print("max |autograd - numeric| =", np.abs(num - W.grad).max())

max |autograd - numeric| = 5.562572624739914e-10


### 3) Train a network with the framework: XOR with `Sequential` + `MSELoss` + `SGD`

In [5]:
np.random.seed(0)
data = Tensor(np.array([[0, 0], [0, 1], [1, 0], [1, 1]]), autograd=True)
target = Tensor(np.array([[0], [1], [1], [0]]), autograd=True)
model = Sequential([Linear(2, 4), Tanh(), Linear(4, 1), Sigmoid()])
criterion, optim = MSELoss(), SGD(model.get_parameters(), alpha=0.5)
for i in range(1000):
    pred = model.forward(data)
    loss = criterion.forward(pred, target)
    loss.backward(np.ones_like(loss.data))
    optim.step()
    if i % 200 == 0: print(f"iter {i:4d} loss {loss.data.sum():.5f}")
print("predictions:", model.forward(data).data.ravel().round(2))

iter    0 loss 1.15611
iter  200 loss 0.04986
iter  400 loss 0.00751
iter  600 loss 0.00383
iter  800 loss 0.00253
predictions: [0.02 0.98 0.98 0.03]


### 4) Embedding + CrossEntropyLoss: learn to predict the next token in a cycle

In [6]:
np.random.seed(1)
vocab_size = 6
inputs  = Tensor(np.array([0, 1, 2, 3, 4, 5]))
targets = Tensor(np.array([1, 2, 3, 4, 5, 0]))      # next token in a ring
embed = Embedding(vocab_size, 8)
model = Sequential([embed, Linear(8, vocab_size)])
crit = CrossEntropyLoss()
optim = SGD(model.get_parameters(), alpha=0.5)
for i in range(201):
    loss = crit.forward(model.forward(inputs), targets)
    loss.backward()
    optim.step()
    if i % 50 == 0: print(f"iter {i:3d} loss {float(loss.data):.4f}")
print("predicted next:", model.forward(inputs).data.argmax(1))

iter   0 loss 1.7821
iter  50 loss 0.0481
iter 100 loss 0.0150
iter 150 loss 0.0083
iter 200 loss 0.0055
predicted next: [1 2 3 4 5 0]


### 5) An RNN cell built from framework layers
Unrolling over time just reuses the same cell (shared weights); autograd handles BPTT automatically.

In [7]:
class RNNCell(Layer):
    def __init__(self, n_in, n_hidden, n_out, activation=Tanh):
        super().__init__()
        self.n_hidden = n_hidden
        self.w_ih = Linear(n_in, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_out)
        self.activation = activation()
        self.parameters = self.w_ih.get_parameters() + self.w_hh.get_parameters() + self.w_ho.get_parameters()
    def forward(self, x, hidden):
        new_h = self.activation.forward(self.w_ih.forward(x) + self.w_hh.forward(hidden))
        return self.w_ho.forward(new_h), new_h
    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)

# Task: from the sequence 0,1,2,3 predict next token (4) - the RNN must use order
np.random.seed(2)
V = 5
embed = Embedding(V, 8); cell = RNNCell(8, 16, V); crit = CrossEntropyLoss()
optim = SGD(embed.get_parameters() + cell.get_parameters(), alpha=0.2)
seq = [0, 1, 2, 3, 4]
for i in range(301):
    h = cell.init_hidden(1); total = None
    for t in range(len(seq) - 1):
        out, h = cell.forward(embed.forward(np.array([seq[t]])), h)
        l = crit.forward(out, np.array([seq[t + 1]]))
        total = l if total is None else total + l
    total.backward(); optim.step()
    if i % 100 == 0: print(f"iter {i:3d} loss {float(total.data):.4f}")
h = cell.init_hidden(1); preds = []
for t in range(len(seq) - 1):
    out, h = cell.forward(embed.forward(np.array([seq[t]])), h); preds.append(int(out.data.argmax()))
print("predictions:", preds, "(expected [1, 2, 3, 4])")

iter   0 loss 6.3810
iter 100 loss 0.0109
iter 200 loss 0.0053
iter 300 loss 0.0034
predictions: [1, 2, 3, 4] (expected [1, 2, 3, 4])


## Takeaways
- **Autograd = remember the graph + apply the chain rule backward.** Only the forward pass needs to be written.
- Layers, losses and optimizers are small classes that compose; this is the same design as PyTorch's `nn` / `optim`.
- With this framework, larger models (LSTMs, next chapter) are just a few lines.